# 02 — Daily Load : validation ETL journalier

Validation des tables produites par `run_daily_load` pour la plage **2026-04-29 → 2026-05-07**.

Tables attendues : `DIM_CITY` (augmentée), `DIM_TRIP`, `FACT_MISSION`, `FACT_EQUIPMENT`.

In [ ]:
import os
import sys
import tomllib
from datetime import date
from pathlib import Path

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Load local path configuration.
# Copy config/config.example.toml → config/config.toml and adjust paths if needed.
_cfg_file = Path("../config/config.toml")
_project_root = _cfg_file.parent.parent

with _cfg_file.open("rb") as _f:
    _cfg = tomllib.load(_f)

_src_path = (_project_root / _cfg["paths"]["src_path"]).resolve()
_data_path = (_project_root / _cfg["paths"]["data_path"]).resolve()

sys.path.insert(0, str(_src_path))

from config.settings import ETLConfig
from utils.spark import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("WARN")
config = ETLConfig(data_path=_data_path)

In [ ]:
from jobs.initial_load import run_initial_load

initial = run_initial_load(spark, config)
print("Initial load OK")
print(f"  DIM_DATE          : {initial.dim_date.count()} lignes")
print(f"  DIM_TRANSPORT_TYPE: {initial.dim_transport_type.count()} lignes")
print(f"  DIM_EQUIPMENT     : {initial.dim_equipment.count()} lignes")
print(f"  DIM_CITY (initial): {initial.dim_city.count()} villes")
print(f"  DIM_STAFF         : {initial.dim_staff.count()} employés")

In [ ]:
from jobs.daily_load import run_daily_load

DATE_DEBUT = date(2026, 4, 29)
DATE_FIN = date(2026, 5, 7)

daily = run_daily_load(spark, config, DATE_DEBUT, DATE_FIN, initial)
print("Daily load OK")

## DIM_CITY augmentée

In [ ]:
print(
    f"DIM_CITY : {daily.dim_city.count()} villes "
    f"(dont {daily.dim_city.filter('IS_ORG_SITE').count()} sites org)"
)
daily.dim_city.filter("IS_ORG_SITE").select("CITY_NAME", "COUNTRY_ISO2").show()

In [ ]:
# Cities without ISO2 country code (unrecognised country name in source files)
no_iso2 = daily.dim_city.filter("COUNTRY_ISO2 IS NULL")
print(f"Villes sans code pays : {no_iso2.count()}")
no_iso2.select("CITY_NAME").show(20)

## DIM_TRIP

In [ ]:
print(f"DIM_TRIP : {daily.dim_trip.count()} trajets distincts")
daily.dim_trip.orderBy("DISTANCE_KM", ascending=False).show(10)

In [ ]:
# Trajets sans distance (villes non géocodées)
from pyspark.sql import functions as F

no_dist = daily.dim_trip.filter("DISTANCE_KM IS NULL")
print(f"Trajets sans distance : {no_dist.count()}")

## FACT_MISSION

In [ ]:
print(f"FACT_MISSION : {daily.fact_mission.count()} missions")
daily.fact_mission.show(5)

In [ ]:
# Répartition par type de mission
daily.fact_mission.groupBy("MISSION_TYPE").count().orderBy(
    "count", ascending=False
).show()

# CO2 total (en tCO2e)
total_co2 = daily.fact_mission.agg(F.sum("CO2_IMPACT_KG").alias("CO2_KG")).collect()[0][
    "CO2_KG"
]
print(f"CO2 total missions : {total_co2 / 1000:.2f} tCO₂e")

In [ ]:
# Missions sans CO2 calculé (distance manquante ou transport non reconnu)
null_co2 = daily.fact_mission.filter("CO2_IMPACT_KG IS NULL")
print(f"Missions sans CO2 : {null_co2.count()}")
null_co2.groupBy("MISSION_TYPE").count().show()

## FACT_EQUIPMENT

In [ ]:
print(f"FACT_EQUIPMENT : {daily.fact_equipment.count()} achats")
daily.fact_equipment.show(5)

total_eq_co2 = daily.fact_equipment.agg(
    F.sum("CO2_IMPACT_KG").alias("CO2_KG")
).collect()[0]["CO2_KG"]
print(f"CO2 total équipements : {total_eq_co2 / 1000:.2f} tCO₂e")

In [ ]:
# Achats sans CO2 (lookup non trouvé même avec fallback)
null_eq = daily.fact_equipment.filter("CO2_IMPACT_KG IS NULL")
print(f"Achats sans CO2 : {null_eq.count()}")